# Phase 4: Methods Parser

This notebook demonstrates the `MethodsParser` — given a `Paper` object, it:

1. Locates and extracts the methods section text (handles all common title variants).
2. Identifies distinct assays and analysis procedures via LLM.
3. Decomposes each assay into ordered `AnalysisStep` objects.
4. Identifies directed dependencies between assays to build an `AssayGraph` DAG.
5. Extracts data- and code-availability statements.

Cells 1–6 require **no API key**. Cells 7–9 mock the LLM. Cell 10 is the live end-to-end demo (requires `ANTHROPIC_API_KEY`).

In [ ]:
import json, os
from unittest.mock import patch, MagicMock

from researcher_ai.models.method import (
    Method, Assay, AnalysisStep, AssayDependency, AssayGraph,
)
from researcher_ai.models.paper import Paper, Section, PaperSource
from researcher_ai.parsers.methods_parser import (
    MethodsParser,
    _extract_section_by_heading,
    _extract_assay_paragraph,
    _build_figure_context,
    _assay_from_meta,
    _METHODS_TITLE_RE,
    _AssayMeta, _StepMeta,
)

print('Imports OK')

## 1. Methods section title detection

`_METHODS_TITLE_RE` matches the many section title variants used across journals:
`Methods`, `Materials and Methods`, `Materials & Methods`, `STAR Methods`, `Online Methods`, `Experimental Procedures`, `Supplementary Methods`.

Non-methods headings (`Results`, `Discussion`, etc.) are correctly rejected.

In [ ]:
# Demonstrate which titles match and which do not
should_match = [
    'Methods', 'METHODS', 'methods',
    'Materials and Methods', 'Materials & Methods',
    'Methods and Materials',
    'Experimental Procedures', 'Experimental Design',
    'STAR Methods', 'STAR* Methods',
    'Online Methods',
    'Supplementary Methods',
]
should_reject = ['Results', 'Discussion', 'Introduction', 'Abstract', 'References']

print('Matches:')
for title in should_match:
    hit = bool(_METHODS_TITLE_RE.search(title))
    mark = '✓' if hit else '✗ FAIL'
    print(f'  {mark}  {title!r}')

print('\nRejections:')
for title in should_reject:
    hit = bool(_METHODS_TITLE_RE.search(title))
    mark = '✓ (no match)' if not hit else '✗ FALSE POSITIVE'
    print(f'  {mark}  {title!r}')

## 2. Methods section extraction

`_extract_methods_text()` searches `paper.sections` by title regex, then falls back
to `paper.methods_section`, then to a raw-text heading scan.

Here we build a minimal `Paper` object and confirm the right section is returned.

In [ ]:
# Paper with a 'Materials and Methods' section
paper = Paper(
    title='Demo paper',
    source=PaperSource.PMID,
    source_path='00000000',
    sections=[
        Section(title='Introduction', text='Background text here.'),
        Section(title='Materials and Methods', text=
            'Cell lines were grown in DMEM. '
            'RNA was extracted using TRIzol. '
            'Libraries were prepared with the TruSeq kit and sequenced on NovaSeq. '
            'Reads were aligned to hg38 with STAR (v2.7.10). '
            'Differential expression was assessed with DESeq2 (v1.38).'
        ),
        Section(title='Results', text='Figure 1 shows alignment rates.'),
    ]
)

parser = MethodsParser()
methods_text = parser._extract_methods_text(paper)
print('Extracted methods text:')
print(methods_text)

## 3. Assay paragraph extraction

`_extract_assay_paragraph()` locates the paragraph(s) in the methods text most
relevant to a given assay name by keyword-scoring each paragraph and returning
the top-3 scoring paragraphs (capped at 3000 chars).

This limits LLM prompt size while keeping context-relevant content.

In [ ]:
multiassay_methods = """
Cell culture and lysis
HEK293T cells were grown in DMEM supplemented with 10% FBS.
Cells were lysed in RIPA buffer and protein concentration measured by BCA assay.

RNA-seq library preparation
Total RNA was extracted using TRIzol reagent. Poly-A selection was performed
using NEBNext Poly(A) mRNA Magnetic Isolation Module. Libraries were prepared
with the NEBNext Ultra II Directional RNA Library Prep Kit.

STAR alignment
Paired-end 150 bp reads were aligned to the GRCh38 genome using STAR v2.7.10a
with the --outSAMtype BAM SortedByCoordinate flag. Alignment rates exceeded 90%.

DESeq2 differential expression
Read counts were generated with featureCounts. Differential expression analysis
was performed in R using DESeq2 v1.38 with Wald test and BH-corrected p-values.
"""

for assay_name in ['RNA-seq library preparation', 'STAR alignment', 'DESeq2']:
    para = _extract_assay_paragraph(multiassay_methods, assay_name)
    preview = para[:120].replace('\n', ' ')
    print(f"Assay: {assay_name!r}")
    print(f"  → {preview!r}")
    print()

## 4. Assay and AssayGraph model construction

`_assay_from_meta()` converts LLM-extracted `_AssayMeta` into typed `Assay` objects.
Multiple `Assay` objects plus `AssayDependency` edges form an `AssayGraph` (DAG).

### AssayGraph helpers

- `get_assay(name)` → `Assay | None`
- `upstream_of(name)` → list of assays that feed this one
- `downstream_of(name)` → list of assays this one feeds

In [ ]:
# Build a 3-step pipeline manually to demonstrate the graph helpers
rnaseq_prep = Assay(
    name='RNA-seq library preparation',
    description='Poly-A selection and library construction from total RNA.',
    data_type='sequencing',
    raw_data_source='GEO: GSE72987',
    steps=[
        AnalysisStep(
            step_number=1,
            description='Extract total RNA with TRIzol.',
            input_data='Frozen cell pellet',
            output_data='Total RNA',
            software='TRIzol',
        ),
        AnalysisStep(
            step_number=2,
            description='Select poly-A mRNA and construct indexed sequencing library.',
            input_data='Total RNA',
            output_data='Sequencing-ready library (FASTQ)',
            software='NEBNext Ultra II',
        ),
    ],
)

star_align = Assay(
    name='STAR alignment',
    description='Align paired-end reads to GRCh38 with STAR.',
    data_type='sequencing',
    steps=[
        AnalysisStep(
            step_number=1,
            description='Align reads to genome.',
            input_data='Raw FASTQ reads',
            output_data='Sorted BAM files',
            software='STAR',
            software_version='2.7.10a',
            parameters={'outSAMtype': 'BAM SortedByCoordinate'},
        ),
    ],
)

deseq2 = Assay(
    name='DESeq2 differential expression',
    description='Differential expression analysis from read counts.',
    data_type='computational',
    steps=[
        AnalysisStep(
            step_number=1,
            description='Count reads per gene.',
            input_data='BAM files',
            output_data='Count matrix',
            software='featureCounts',
        ),
        AnalysisStep(
            step_number=2,
            description='Fit negative binomial GLM and compute Wald test statistics.',
            input_data='Count matrix',
            output_data='DE results table (log2FC, BH-adjusted p-values)',
            software='DESeq2',
            software_version='1.38',
        ),
    ],
    figures_produced=['Figure 3', 'Figure 4a'],
)

# DAG edges: prep → alignment → DE
graph = AssayGraph(
    assays=[rnaseq_prep, star_align, deseq2],
    dependencies=[
        AssayDependency(
            upstream_assay='RNA-seq library preparation',
            downstream_assay='STAR alignment',
            dependency_type='raw_data',
            description='FASTQ files from library prep are aligned.',
        ),
        AssayDependency(
            upstream_assay='STAR alignment',
            downstream_assay='DESeq2 differential expression',
            dependency_type='count_input',
            description='BAM files are counted to produce the DESeq2 count matrix.',
        ),
    ],
)

print('AssayGraph summary')
print(f'  Assays : {[a.name for a in graph.assays]}')
print(f'  Edges  : {len(graph.dependencies)}')
print()

print('Graph helper queries:')
print(f'  get_assay("STAR alignment") → {graph.get_assay("STAR alignment").name}')
print(f'  upstream_of("STAR alignment") → {graph.upstream_of("STAR alignment")}')
print(f'  downstream_of("STAR alignment") → {graph.downstream_of("STAR alignment")}')
print(f'  upstream_of("RNA-seq library preparation") → {graph.upstream_of("RNA-seq library preparation")}')

## 5. Assay identification — mocked LLM

`_identify_assays()` prompts the LLM with the full methods text and returns a flat
list of assay name strings. Here we mock the LLM response to verify the integration
without a live API key.

In [ ]:
from researcher_ai.parsers.methods_parser import _AssayList

_mock_assay_list = _AssayList(assay_names=[
    'RNA-seq library preparation',
    'STAR alignment',
    'DESeq2 differential expression',
])

with patch(
    'researcher_ai.parsers.methods_parser.ask_claude_structured',
    return_value=_mock_assay_list,
):
    parser2 = MethodsParser()
    names = parser2._identify_assays(multiassay_methods)

print('Identified assays:')
for n in names:
    print(f'  - {n}')

## 6. Full parse() — mocked LLM

`MethodsParser.parse(paper)` orchestrates all the above steps and returns a
complete `Method` object. We mock every LLM call to exercise the integration
path without a live API key.

This also validates that per-assay failures produce stubs rather than aborting
the entire parse (the pipeline resilience guarantee).

In [ ]:
from researcher_ai.parsers.methods_parser import (
    _AssayList, _AssayMeta, _StepMeta, _DependencyList, _AvailabilityStatement,
)

# ---- Mock responses ----
_mock_assay_list = _AssayList(assay_names=[
    'RNA-seq library preparation',
    'STAR alignment',
    'DESeq2 differential expression',
])

_mock_assay_meta = _AssayMeta(
    name='RNA-seq library preparation',
    description='Total RNA extraction and library preparation.',
    data_type='sequencing',
    raw_data_source='GEO: GSE72987',
    steps=[
        _StepMeta(
            step_number=1,
            description='Extract RNA with TRIzol.',
            input_data='Cell pellet',
            output_data='Total RNA',
        ),
    ],
)

_mock_deps = _DependencyList(dependencies=[])
_mock_avail = _AvailabilityStatement(
    data_statement='Sequencing data deposited at GEO under GSE72987.',
    code_statement='Analysis scripts available at github.com/lab/rnaseq-pipeline.',
)

_call_count = [0]
def _side_effect(prompt, output_schema, **kw):
    _call_count[0] += 1
    if output_schema is _AssayList:
        return _mock_assay_list
    if output_schema is _AssayMeta:
        return _mock_assay_meta
    if output_schema is _DependencyList:
        return _mock_deps
    if output_schema is _AvailabilityStatement:
        return _mock_avail
    raise ValueError(f'Unexpected schema: {output_schema}')

paper_full = Paper(
    title='Demo RNA-seq paper',
    doi='10.1234/demo',
    source=PaperSource.DOI,
    source_path='10.1234/demo',
    sections=[
        Section(title='Materials and Methods', text=multiassay_methods),
    ]
)

with patch(
    'researcher_ai.parsers.methods_parser.ask_claude_structured',
    side_effect=_side_effect,
):
    _call_count[0] = 0
    parser3 = MethodsParser()
    method = parser3.parse(paper_full)

print(f'Total LLM calls: {_call_count[0]}')
print(f'paper_doi       : {method.paper_doi}')
print(f'Assays parsed   : {len(method.assay_graph.assays)}')
for a in method.assay_graph.assays:
    print(f'  - {a.name} ({len(a.steps)} steps)')
print(f'Dependencies    : {len(method.assay_graph.dependencies)}')
print(f'Data avail.     : {method.data_availability!r}')
print(f'Code avail.     : {method.code_availability!r}')

## 7. Graceful failure: stub assay on parse error

When `_parse_assay()` raises for a specific assay, `parse()` catches the exception,
appends a stub `Assay` with `description='Could not be parsed.'`, and continues to
the next assay. The overall `Method` is still returned.

This is the same pipeline resilience pattern used by `FigureParser`.

In [ ]:
fail_count = [0]
def _failing_assay_side_effect(prompt, output_schema, **kw):
    if output_schema is _AssayList:
        return _AssayList(assay_names=['Good Assay', 'Bad Assay'])
    if output_schema is _AssayMeta:
        fail_count[0] += 1
        if fail_count[0] == 1:
            return _AssayMeta(
                name='Good Assay', description='Works fine.',
                data_type='sequencing', steps=[],
            )
        raise RuntimeError('LLM timed out')
    if output_schema is _DependencyList:
        return _DependencyList(dependencies=[])
    return _AvailabilityStatement()

with patch(
    'researcher_ai.parsers.methods_parser.ask_claude_structured',
    side_effect=_failing_assay_side_effect,
):
    fail_count[0] = 0
    method_partial = MethodsParser().parse(paper_full)

print(f'Assays returned : {len(method_partial.assay_graph.assays)}')
for a in method_partial.assay_graph.assays:
    print(f'  - {a.name!r}: {a.description!r}')

## 8. JSON round-trip verification

All `Method` → JSON → `Method` transformations must be lossless (Pydantic v2).
This cell serialises the `method` object from cell 6 and validates that every
field survives the round-trip.

In [ ]:
json_str = method.model_dump_json(indent=2)
method_restored = Method.model_validate_json(json_str)

assert method_restored.paper_doi == method.paper_doi
assert len(method_restored.assay_graph.assays) == len(method.assay_graph.assays)
assert method_restored.data_availability == method.data_availability
assert method_restored.code_availability == method.code_availability

print('Round-trip verified ✓')
print()
print('JSON preview (first 600 chars):')
print(json_str[:600])

## 9. Live parse from PaperParser output (end-to-end demo)

Full pipeline: `PaperParser` → `FigureParser` → `MethodsParser`.
Requires `ANTHROPIC_API_KEY`. Skip if key is absent.

In [ ]:
if not os.environ.get('ANTHROPIC_API_KEY'):
    print('ANTHROPIC_API_KEY not set — skipping live demo.')
else:
    from researcher_ai.parsers.paper_parser import PaperParser
    from researcher_ai.parsers.figure_parser import FigureParser

    # eCLIP paper (PMID 26971820) — has a rich Methods section
    pp = PaperParser()
    paper_live = pp.parse('26971820')

    fp = FigureParser()
    figures_live = fp.parse_all_figures(paper_live)

    mp = MethodsParser()
    method_live = mp.parse(paper_live, figures=figures_live)

    print(f'Paper : {paper_live.title[:60]}')
    print(f'Assays found : {len(method_live.assay_graph.assays)}')
    for a in method_live.assay_graph.assays:
        print(f'  [{a.data_type:12s}] {a.name} — {len(a.steps)} steps')
    print(f'Dependencies : {len(method_live.assay_graph.dependencies)}')
    print(f'Data avail.  : {method_live.data_availability[:80]!r}')

## Summary

Phase 4 delivers:

- **`MethodsParser.parse(paper, figures=None)`** — orchestrates full methods-to-`Method` pipeline.
- **Methods section extraction** — regex-based title matching (13 variants), fallback to raw text scan.
- **Assay identification** — LLM extracts a flat list of distinct assays/procedures.
- **Per-assay step decomposition** — keyword-scored paragraph selection feeds LLM; each assay
  becomes an `Assay` with ordered `AnalysisStep` objects (software, version, parameters, I/O).
- **AssayGraph DAG** — `AssayDependency` edges between assays; `upstream_of` / `downstream_of`
  helpers for traversal.
- **Availability statement extraction** — data and code availability parsed from dedicated
  sections or inline methods text.
- **Graceful failure** — per-assay LLM failures produce stubs without aborting the parse.
- **JSON round-trip** — all `Method`, `Assay`, `AnalysisStep`, `AssayGraph` models
  serialise and deserialise losslessly via Pydantic v2.